## Cell 1: Setup

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil, zipfile
MYDRIVE = '/content/drive/MyDrive'
WORK = f'{MYDRIVE}/gan_vs_diffusion_work'; os.makedirs(WORK, exist_ok=True)
CODE = '/content/gan_vs_diffusion'
DATA = '/content/bbbc_128'

if not os.path.exists(f'{CODE}/ddpm/train.py'):
    z = f'{MYDRIVE}/gan_vs_diffusion.zip'
    if not os.path.exists(z):                      # e.g. "gan_vs_diffusion (1).zip"
        found = sorted(glob.glob(f'{MYDRIVE}/gan_vs_diffusion*.zip'))
        if not found:
            raise RuntimeError('Put gan_vs_diffusion.zip directly in My Drive, then run again.')
        z = found[0]
    shutil.rmtree('/content/_code', ignore_errors=True)
    zipfile.ZipFile(z).extractall('/content/_code')
    src = glob.glob('/content/_code/**/ddpm/train.py', recursive=True)[0]
    shutil.copytree(os.path.dirname(os.path.dirname(src)), CODE, dirs_exist_ok=True)
os.chdir(CODE)
!pip install -q tifffile torch-fidelity ninja click psutil imageio scikit-image
!rm -rf third_party/stylegan3          # always start from a fresh copy of NVIDIA's code
!python stylegan/setup_stylegan3.py

# Compatibility fixes for NVIDIA's code on current Colab (safe to run more than once)
SG = f'{CODE}/third_party/stylegan3'
p = f'{SG}/torch_utils/misc.py'
s = open(p).read()
open(p, 'w').write(s.replace('super().__init__(dataset)', 'super().__init__()'))
FIX = """

# gan_vs_diffusion: fall back to the reference implementation
_gvd_init_orig = _init
_gvd_failed = False

def _init():
    global _gvd_failed
    if _gvd_failed:
        return False
    try:
        return _gvd_init_orig()
    except Exception as e:
        _gvd_failed = True
        print(f'Note: custom CUDA kernel unavailable ({e.__class__.__name__}); using the reference implementation.')
        return False
"""
for name in ['bias_act.py', 'upfirdn2d.py', 'filtered_lrelu.py']:
    p = f'{SG}/torch_utils/ops/{name}'
    s = open(p).read()
    if 'gan_vs_diffusion: fall back' not in s:
        open(p, 'w').write(s + FIX)
# check NVIDIA's code is complete before training
import py_compile
assert 'def profiled_function' in open(f'{SG}/torch_utils/misc.py').read(), 'misc.py damaged'
for rel in ['torch_utils/misc.py', 'torch_utils/ops/bias_act.py', 'torch_utils/ops/upfirdn2d.py',
            'torch_utils/ops/filtered_lrelu.py', 'dnnlib/util.py', 'train.py']:
    py_compile.compile(f'{SG}/{rel}', doraise=True)
print('Setup done:', CODE)

## Cell 2: Unpacking prepared data from Drive

In [ ]:
if not os.path.exists(f'{DATA}/manifest.json'):
    zp = f'{WORK}/bbbc_128.zip'
    if not os.path.exists(zp):
        raise RuntimeError('bbbc_128.zip not found in gan_vs_diffusion_work - upload it from your Mac, then run again.')
    !unzip -oq {zp} -d /content
!ls {DATA} {DATA}/subsets/seed0

## Cell 3: Training


In [ ]:
!python stylegan/train_stylegan2ada.py --data-root {DATA} --outdir {WORK}/runs/stylegan2ada \
    --mode scratch --sizes 5 25 100 --seeds 0 --kimg 300 --snap-kimg 50 --batch-gpu 32
if _exit_code != 0:
    raise RuntimeError('StyleGAN training failed - see the error above (the notebook stops here).')

## Cell 4: Evaluation


In [ ]:
VAL, TEST = f'{DATA}/bbbc039/validation/images', f'{DATA}/bbbc039/test/images'
for n in [5, 25, 100]:
    !python eval/select_best.py --kind stylegan --run {WORK}/runs/stylegan2ada/scratch-n{n}-seed0 \
        --real {VAL} --real-final {TEST} --n-screen 500 --n-final 1000 --every 2
    if _exit_code != 0:
        raise RuntimeError(f'Evaluation of n={n} failed - see the error above (the notebook stops here).')

## Cell 5: Results

In [ ]:
import json
ok = True
for n in [5, 25, 100]:
    f = f'{WORK}/runs/stylegan2ada/scratch-n{n}-seed0/final_metrics.json'
    if os.path.exists(f):
        m = json.load(open(f))
        print(f'StyleGAN n={n:<4}', {k: round(v, 4) for k, v in m.items() if isinstance(v, float)})
    else:
        ok = False; print(f'StyleGAN n={n:<4} not finished')

## Switch the GPU off

In [ ]:
if ok:
    from google.colab import runtime
    runtime.unassign()
else:
    print('Not everything finished - GPU left on so you can check. Disconnect manually if you stop.')